# Fine-tune PhoBERT cho Intent Classification
- **Môi trường**: Google Colab
- **Phần cứng**: 1x T4 GPU
- **Model**: vinai/phobert-base-v2
- **Dataset**: intent.csv (1 cột `text`, 1 cột `label`)

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Load secret từ Google Colab (cần thêm secret tên hf_token trong tab hình chiếc chìa khóa bên trái)
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

In [ ]:
import pandas as pd
from datasets import Dataset

df = pd.read_csv('intent.csv')

# Ánh xạ label dạng text sang dạng số (ID)
label2id = {label: i for i, label in enumerate(df['label'].unique())}
id2label = {i: label for label, i in label2id.items()}

df['label'] = df['label'].map(label2id)

# Chuyển DataFrame sang HuggingFace Dataset và chia train/test (90/10)
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "vinai/phobert-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=256)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Tải pre-trained model và chỉ định số lượng class
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

Map:   0%|          | 0/36000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
hf_token = userdata.get('HF_TOKEN_WRITE')
login(token=hf_token)

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="phobert-intent-classifier",
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    lr_scheduler_type="linear",
    warmup_steps=200,
    learning_rate=1e-4,
    per_device_train_batch_size=96,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    push_to_hub=True,
    fp16=True, # Bật FP16 giúp train nhanh hơn và ít tốn VRAM trên T4
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    compute_metrics=compute_metrics
)

In [ ]:
# Bắt đầu huấn luyện
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.186253,0.000964,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.186253,0.000964,1.000000
2,0.004284,0.000358,1.000000
3,0.001335,0.000164,1.000000
4,0.000234,0.000112,1.000000
5,0.000163,0.000100,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1875, training_loss=0.03845372103651365, metrics={'train_runtime': 2311.6993, 'train_samples_per_second': 77.865, 'train_steps_per_second': 0.811, 'total_flos': 2.368042020864e+16, 'train_loss': 0.03845372103651365, 'epoch': 5.0})

In [ ]:
trainer.save_model("phobert-intent-classifier")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
!pip -q install optimum[onnxruntime] onnx onnxruntime

## Restart session here

In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

model_id = "./phobert-intent-classifier" # Đường dẫn model của bạn
onnx_path = "./phobert-intent-onnx"      # Thư mục lưu ONNX

# SỬA Ở ĐÂY: Load tokenizer thẳng từ gốc của PhoBERT-base-v2
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

# Load model Pytorch từ thư mục local và convert sang ONNX
model = ORTModelForSequenceClassification.from_pretrained(model_id, export=True)

# Lưu cả model ONNX và tokenizer vào thư mục đích
tokenizer.save_pretrained(onnx_path)
model.save_pretrained(onnx_path)

print(f"Đã xuất mô hình ONNX thành công tại: {onnx_path}")

Multiple distributions found for package optimum. Picked distribution: optimum
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


Đã xuất mô hình ONNX thành công tại: ./phobert-intent-onnx
